In [1]:
import pandas as pd

df_sales = pd.read_csv(
    "sales_dataset_hakaton_train.csv",
    parse_dates=["date"]
)

df_stock = pd.read_csv(
    "stock_dataset_hakaton_train.csv",
    parse_dates=["date"]
)

df_sales.head()

,date,composite_division,cities_name,warehouse_name,category_lvl_1,category_lvl_2,category_lvl_3,article,characteristic,quantity,sales_sum,discount_sum
0,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_0,category_lvl_2_0,NaN,char_article_0,characteristic_0,2.0,7.429034,5.459302
1,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_1,category_lvl_2_1,NaN,char_article_1,characteristic_0,1.0,9.613652,7.798864
2,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_2,category_lvl_2_2,NaN,char_article_2,characteristic_0,1.0,9.237081,7.258904
3,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_3,category_lvl_2_3,NaN,char_article_3,characteristic_1,1.0,9.066837,7.094100
4,2024-01-01,composite_division_0,cities_name_1,warehouse_name_1,category_lvl_1_2,category_lvl_2_2,NaN,char_article_4,characteristic_0,1.0,9.237083,7.258905


In [3]:
keys = ["date", "warehouse_name", "article"]

df_sales_clean = (
    df_sales
    .groupby(keys, as_index=False)
    .agg(quantity=("quantity", "sum"))
)

df_sales_clean.head()

,date,warehouse_name,article,quantity
0,2024-01-01,warehouse_name_0,char_article_0,2.0
1,2024-01-01,warehouse_name_0,char_article_1,1.0
2,2024-01-01,warehouse_name_0,char_article_2,2.0
3,2024-01-01,warehouse_name_0,char_article_3,1.0
4,2024-01-01,warehouse_name_0,char_article_73,1.0


In [5]:
stock_duplicates = df_stock.duplicated(keys, keep=False)
print("Дубликатов в остатках:", stock_duplicates.sum())

Дубликатов в остатках: 2309095


In [6]:
if stock_duplicates.any():
    raise ValueError(
        "В таблице остатков есть дубликаты. "
        "Нужно выяснить их причину."
    )

ValueError: В таблице остатков есть дубликаты. Нужно выяснить их причину.

In [7]:
keys = ["date", "warehouse_name", "article"]

print("Всего строк:", len(df_stock))
print("Уникальных комбинаций:", len(df_stock[keys].drop_duplicates()))

print("\nПример повторяющихся комбинаций:")
print(
    df_stock[df_stock.duplicated(keys, keep=False)]
    .sort_values(keys)
    .head(20)
)


Всего строк: 6153988
Уникальных комбинаций: 4884198

Пример повторяющихся комбинаций:
            date    warehouse_name           article  stock
51645 2024-01-17  warehouse_name_2  char_article_100      8
52058 2024-01-17  warehouse_name_2  char_article_100    295
52376 2024-01-17  warehouse_name_2  char_article_101      0
52785 2024-01-17  warehouse_name_2  char_article_101      0
52390 2024-01-17  warehouse_name_2  char_article_104      1
52852 2024-01-17  warehouse_name_2  char_article_104      7
51837 2024-01-17  warehouse_name_2  char_article_107    444
52757 2024-01-17  warehouse_name_2  char_article_107      1
51528 2024-01-17  warehouse_name_2  char_article_108      2
52566 2024-01-17  warehouse_name_2  char_article_108    294
52388 2024-01-17  warehouse_name_2  char_article_111     11
52529 2024-01-17  warehouse_name_2  char_article_111      1
51754 2024-01-17  warehouse_name_2  char_article_129     30
52098 2024-01-17  warehouse_name_2  char_article_129      3
51553 2024-01-

In [8]:
example = (
    df_stock
    .groupby(keys, dropna=False)
    .agg(
        rows=("stock", "size"),
        unique_stock=("stock", "nunique"),
        min_stock=("stock", "min"),
        max_stock=("stock", "max")
    )
    .sort_values("rows", ascending=False)
)

print(example.head(20))

                                              rows  unique_stock  min_stock  \
date       warehouse_name   article                                           
2026-07-03 warehouse_name_2 char_article_29      4             3          0   
2026-07-10 warehouse_name_2 char_article_96      4             3          0   
2026-05-27 warehouse_name_2 char_article_555     4             3          0   
                            char_article_556     4             3          0   
                            char_article_557     4             1          0   
                            char_article_558     4             3          0   
                            char_article_56      4             3          0   
                            char_article_571     4             2          0   
                            char_article_573     4             3          0   
                            char_article_574     4             2          0   
                            char_article_575     4  

In [9]:
print(df_stock.columns.tolist())
print(df_stock.dtypes)
print(df_stock.head(10).to_string(index=False))


['date', 'warehouse_name', 'article', 'stock']
date              datetime64[ns]
warehouse_name            object
article                   object
stock                      int64
dtype: object
      date   warehouse_name          article  stock
2023-12-13 warehouse_name_0 char_article_119      1
2023-12-13 warehouse_name_3  char_article_62    135
2023-12-13 warehouse_name_3 char_article_268      3
2023-12-13 warehouse_name_7 char_article_258      1
2023-12-13 warehouse_name_3  char_article_14     19
2023-12-13 warehouse_name_2 char_article_360      6
2023-12-13 warehouse_name_2 char_article_117      7
2023-12-13 warehouse_name_6  char_article_28    160
2023-12-13 warehouse_name_2  char_article_94    142
2023-12-13 warehouse_name_3 char_article_272      1


In [10]:
# Сколько раз повторяется каждый ключ
key_counts = df_stock.groupby(keys).size()

print("Распределение количества записей на ключ:")
print(key_counts.value_counts().sort_index())

print("\nКоличество строк по датам:")
print(df_stock.groupby("date").size().sort_index().to_string())

print("\nКоличество уникальных складов:")
print(df_stock["warehouse_name"].nunique())

print("\nКоличество уникальных товаров:")
print(df_stock["article"].nunique())

Распределение количества записей на ключ:
1    3844893
2     826839
3     194447
4      18019
Name: count, dtype: int64

Количество строк по датам:
date
2023-12-13     1066
2023-12-14     1141
2023-12-15     1264
2023-12-16     1277
2023-12-17     1293
2023-12-18     1336
2023-12-19     1337
2023-12-20     1337
2023-12-21     1337
2023-12-22     1454
2023-12-23     1465
2023-12-24     1473
2023-12-25     1486
2023-12-26     1496
2023-12-27     1509
2023-12-28     1513
2023-12-29     1525
2023-12-30     1529
2023-12-31     1529
2024-01-01     1530
2024-01-02     1531
2024-01-03     1531
2024-01-04     1531
2024-01-05     1531
2024-01-06     1532
2024-01-07     1532
2024-01-08     1532
2024-01-09     1537
2024-01-10     1553
2024-01-11     1558
2024-01-12     1576
2024-01-13     1578
2024-01-14     1584
2024-01-15     1597
2024-01-16     1604
2024-01-17     1672
2024-01-18     1696
2024-01-19     1719
2024-01-20     1722
2024-01-21     1727
2024-01-22     1739
2024-01-23     1744
2024-01

In [11]:
mask = (
    (df_stock["date"] == "2024-01-17")
    & (df_stock["warehouse_name"] == "warehouse_name_2")
    & (df_stock["article"] == "char_article_100")
)

print(df_stock.loc[mask].to_string(index=False))

      date   warehouse_name          article  stock
2024-01-17 warehouse_name_2 char_article_100      8
2024-01-17 warehouse_name_2 char_article_100    295


In [12]:
print(
    "Продажи:",
    df_sales["date"].min(),
    "—",
    df_sales["date"].max()
)

print(
    "Остатки:",
    df_stock["date"].min(),
    "—",
    df_stock["date"].max()
)

Продажи: 2024-01-01 00:00:00 — 2026-07-31 00:00:00
Остатки: 2023-12-13 00:00:00 — 2026-07-31 00:00:00


In [13]:
keys = ["date", "warehouse_name", "article"]

print(
    "Полностью одинаковых строк:",
    df_stock.duplicated(
        ["date", "warehouse_name", "article", "stock"]
    ).sum()
)

print(
    "Групп с разными значениями stock:",
    (
        df_stock.groupby(keys)["stock"]
        .nunique()
        .gt(1)
        .sum()
    )
)

Полностью одинаковых строк: 427990
Групп с разными значениями stock: 804588


In [ ]:
df_stock_clean = df_stock.drop_duplicates(
    subset=["date", "warehouse_name", "article", "stock"]
).copy()

# Агрегация остатков
df_stock_clean = df_stock_clean.groupby(
    keys, as_index=False
).agg(
    stock=("stock", "sum")
)

# Проверка
print("Строк после агрегации:", len(df_stock_clean))
print("Дубликатов осталось:", df_stock_clean.duplicated(keys).sum())

Строк после агрегации: 4884198
Дубликатов осталось: 0


In [15]:
min_date = min(df_stock_clean['date'].min(), df_sales['date'].min())
max_date = max(df_stock_clean['date'].max(), df_sales['date'].max())
all_dates = pd.date_range(min_date, max_date, freq='D')

warehouses = df_sales['warehouse_name'].unique()
articles = df_sales['article'].unique()

In [ ]:
full_grid = pd.MultiIndex.from_product(
    [warehouses, articles, all_dates],
    names=['warehouse_name', 'article', 'date']
).to_frame(index=False)

# Присоединение остатков к сетке
df_stock_daily = full_grid.merge(
    df_stock_clean,
    on=keys,
    how='left'
)


In [17]:
df_stock_daily = df_stock_daily.sort_values(['warehouse_name', 'article', 'date'])
df_stock_daily['stock'] = (
    df_stock_daily.groupby(['warehouse_name', 'article'])['stock']
    .ffill()
    .fillna(0) # 0 ставим только если до этого вообще не было остатков
)

print("Размер ежедневного датасета остатков:", df_stock_daily.shape)
print("Пропусков в stock:", df_stock_daily['stock'].isna().sum())

Размер ежедневного датасета остатков: (8455018, 4)
Пропусков в stock: 0


In [19]:
df_stock_daily.head(200)

,warehouse_name,article,date,stock
0,warehouse_name_0,char_article_0,2023-12-13,1.0
1,warehouse_name_0,char_article_0,2023-12-14,1.0
2,warehouse_name_0,char_article_0,2023-12-15,1.0
3,warehouse_name_0,char_article_0,2023-12-16,0.0
4,warehouse_name_0,char_article_0,2023-12-17,0.0
...,...,...,...,...
195,warehouse_name_0,char_article_0,2024-06-25,0.0
196,warehouse_name_0,char_article_0,2024-06-26,0.0
197,warehouse_name_0,char_article_0,2024-06-27,0.0
198,warehouse_name_0,char_article_0,2024-06-28,0.0


In [20]:
expected_rows = 11 * 799 * len(all_dates) # all_dates мы создавали при построении сетки
print(f"Ожидалось строк: {expected_rows}")
print(f"Фактически:      {len(df_stock_daily)}")

Ожидалось строк: 8455018
Фактически:      8455018


In [21]:
dups = df_stock_daily.duplicated(subset=['date', 'warehouse_name', 'article']).sum()
print(f"Дубликатов: {dups} (должно быть 0)")

Дубликатов: 0 (должно быть 0)


In [ ]:
# проверка пропусков
print(f"Пропусков (NaN) в stock: {df_stock_daily['stock'].isna().sum()} (должно быть 0)")


Пропусков (NaN) в stock: 0 (должно быть 0)


In [23]:
sample = df_stock_daily[
    (df_stock_daily['warehouse_name'] == 'warehouse_name_0') & 
    (df_stock_daily['article'] == 'char_article_0')
].sort_values('date')

In [24]:
date_diffs = sample['date'].diff().dropna().dt.days
print(f"Максимальный разрыв между датами: {date_diffs.max()} дней (должно быть 1)")

Максимальный разрыв между датами: 1 дней (должно быть 1)


In [ ]:
import pandas as pd
import numpy as np

keys = ["date", "warehouse_name", "article"]

# Чистим пропуски в категориях (только текстовые колонки)
cat_cols = ["cities_name", "category_lvl_1", "category_lvl_2", "category_lvl_3"]
for col in cat_cols:
    if col in df_sales.columns:
        df_sales[col] = df_sales[col].fillna("unknown")

# Проверяем на отрицательные продажи
neg = (df_sales["quantity"] < 0).sum()
print(f"Строк с отрицательным quantity: {neg}")

Строк с отрицательным quantity: 0


In [28]:
#  Агрегируем департаменты 
df_sales_clean = (
    df_sales
    .groupby(keys, as_index=False)
    .agg(
        quantity=("quantity", "sum"),
        
        cities_name=("cities_name", "first"),
        category_lvl_1=("category_lvl_1", "first"),
        category_lvl_2=("category_lvl_2", "first"),
        category_lvl_3=("category_lvl_3", "first"),
    )
)

In [ ]:
# Проверка
print("Дубликатов после groupby:", df_sales_clean.duplicated(keys).sum())  # должно быть 0
print("Пропусков в quantity:", df_sales_clean["quantity"].isna().sum())   # должно быть 0
print("Мин. quantity:", df_sales_clean["quantity"].min())                  # должно быть >= 0

Дубликатов после groupby: 0
Пропусков в quantity: 0
Мин. quantity: 1.0


In [ ]:
# проверка

print(f"4. Пропусков в категориях:")
for col in ['cities_name', 'category_lvl_1', 'category_lvl_2', 'category_lvl_3']:
    print(f"   - {col}: {df_sales_clean[col].isna().sum()}")


print(f"\n5. Итоги:")
print(f"   - Строк в очищенном sales: {len(df_sales_clean)}")
print(f"   - Уникальных дат: {df_sales_clean['date'].nunique()}")
print(f"   - Уникальных складов: {df_sales_clean['warehouse_name'].nunique()}")
print(f"   - Уникальных товаров: {df_sales_clean['article'].nunique()}")

4. Пропусков в категориях:
   - cities_name: 0
   - category_lvl_1: 0
   - category_lvl_2: 0
   - category_lvl_3: 0

5. Итоги:
   - Строк в очищенном sales: 970677
   - Уникальных дат: 943
   - Уникальных складов: 11
   - Уникальных товаров: 799


In [ ]:
keys = ["date", "warehouse_name", "article"]
cat_cols = ["cities_name", "category_lvl_1", "category_lvl_2", "category_lvl_3"]

# Присоединяем продажи к сетке стоков
df_final = df_stock_daily.merge(
    df_sales_clean,
    on=keys,
    how="left" # left, потому что df_stock_daily — это полный календарь
)

# Заполняем продажи нулями (для дней, когда товар не продавался)
df_final["quantity"] = df_final["quantity"].fillna(0)

# Заполняем категории (для товаров, которые есть на складе, но ни разу не продавались)
for col in cat_cols:
    df_final[col] = df_final[col].fillna("unknown")

In [32]:

print(f"Размер таблицы: {df_final.shape}")
print(f"Пропусков в quantity: {df_final['quantity'].isna().sum()} (должно быть 0)")
print(f"Пропусков в stock: {df_final['stock'].isna().sum()} (должно быть 0)")
print(f"Дубликатов по ключам: {df_final.duplicated(subset=keys).sum()} (должно быть 0)")

Размер таблицы: (8455018, 9)
Пропусков в quantity: 0 (должно быть 0)
Пропусков в stock: 0 (должно быть 0)
Дубликатов по ключам: 0 (должно быть 0)


In [38]:
# Считаем количество уникальных дней в истории
n_dates = df_final['date'].nunique()
n_warehouses = df_final['warehouse_name'].nunique()
n_articles = df_final['article'].nunique()

# Считаем ожидаемое количество строк
expected_rows = n_warehouses * n_articles * n_dates

print(f"Уникальных складов: {n_warehouses} (по ТЗ должно быть 11)")
print(f"Уникальных артикулов: {n_articles} (по ТЗ должно быть 799)")
print(f"Уникальных дней истории: {n_dates}")

print(f"\nОжидаемое количество строк (11 * 799 * {n_dates}): {expected_rows}")
print(f"Фактическое количество строк: {len(df_final)}")

if len(df_final) == expected_rows:
    print("\nКоличество строк совпадает. Сетка полная, дубликатов и пропусков нет.")
else:
    print(f"\nРазница: {expected_rows - len(df_final)} строк. Где-то потерялись данные.")

Уникальных складов: 11 (по ТЗ должно быть 11)
Уникальных артикулов: 799 (по ТЗ должно быть 799)
Уникальных дней истории: 962

Ожидаемое количество строк (11 * 799 * 962): 8455018
Фактическое количество строк: 8455018

Количество строк совпадает. Сетка полная, дубликатов и пропусков нет.


In [39]:
# Проверка непрерывности дат
dates = pd.Series(sorted(df_final['date'].unique()))
gaps = dates.diff().dt.days.dropna()

print(f"Максимальный разрыв между датами: {gaps.max()} день (должен быть 1)")
print(f"Минимальная дата: {dates.min().date()}")
print(f"Максимальная дата: {dates.max().date()}")

Максимальный разрыв между датами: 1.0 день (должен быть 1)
Минимальная дата: 2023-12-13
Максимальная дата: 2026-07-31


In [ ]:
import os


save_path = r"C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\stock_daily.parquet"

# Сохраняем
df_stock_daily.to_parquet(save_path, index=False)

print(f"✅ Файл успешно сохранен по пути: {save_path}")

✅ Файл успешно сохранен по пути: C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\stock_daily.parquet


In [36]:
import os


save_path = r"C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\df_final.parquet"

# Сохраняем
df_final.to_parquet(save_path, index=False)

print(f"✅ Файл успешно сохранен по пути: {save_path}")

✅ Файл успешно сохранен по пути: C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\df_final.parquet


In [37]:
import os


save_path = r"C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\df_sales_clean.parquet"

# Сохраняем
df_sales_clean.to_parquet(save_path, index=False)

print(f"✅ Файл успешно сохранен по пути: {save_path}")

✅ Файл успешно сохранен по пути: C:\Users\evkho\OneDrive\Рабочий стол\hakaton_datasets\df_sales_clean.parquet
